# Validação reproduzível da IA dos sensores

Este notebook avalia o Isolation Forest com divisão temporal, perturbações sintéticas por intensidade e critérios definidos previamente. Os resultados não possuem significado sanitário e não alteram o modelo epidemiológico Poisson.

## 1. Preparar o ambiente

Execute a célula abaixo uma vez. Ela baixa exclusivamente a branch experimental.

In [ ]:
from pathlib import Path

REPO_DIR = Path('/content/tcc-vigilancia-wbe-iot')
if not REPO_DIR.exists():
    !git clone -q --branch feature/iot-anomaly-ai --single-branch https://github.com/samuelfzst-bit/tcc-vigilancia-wbe-iot.git $REPO_DIR
else:
    !git -C $REPO_DIR fetch -q origin feature/iot-anomaly-ai
    !git -C $REPO_DIR switch -q feature/iot-anomaly-ai
    !git -C $REPO_DIR pull -q --ff-only origin feature/iot-anomaly-ai
%cd /content/tcc-vigilancia-wbe-iot
!python -m pip install -q -r requirements/frontend.txt
print('Ambiente preparado na branch feature/iot-anomaly-ai.')

## 2. Conectar ao Supabase em modo somente leitura

Crie o segredo `TCC_DB_URL` no Colab. A URI não será exibida. Use o usuário de leitura do dashboard, nunca `postgres` ou `service_role`.

In [ ]:
from google.colab import userdata
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from frontend.data_access import database_url
from src.sensor_validation import (
    acceptance_criteria, evaluate_perturbation_grid, temporal_validation,
)

try:
    DB_URL = userdata.get('TCC_DB_URL')
except Exception as exc:
    raise RuntimeError('Crie e habilite o segredo TCC_DB_URL no Colab.') from exc
SAFE_URL = database_url({'TCC_DB_URL': DB_URL})
engine = create_engine(SAFE_URL, pool_pre_ping=True, connect_args={'connect_timeout': 15})
with engine.connect() as connection:
    connection.execute(text('select 1'))
print('Conexão de leitura validada. A credencial não foi exibida.')

In [ ]:
QUERY = text('''
select id, device_id, data_hora, ph_value, turbidity_value,
       tds_value, temperature_value, status
from public.leituras_sensor
where data_hora >= timestamptz '2026-01-01'
order by data_hora, id
''')
with engine.connect() as connection:
    leituras = pd.read_sql(QUERY, connection, parse_dates=['data_hora'])
print('Dimensão carregada:', leituras.shape)
display(leituras.head())

## 3. Auditoria antes do treino

Esta célula apenas descreve os dados. Nenhuma ausência é substituída por zero.

In [ ]:
sensor_cols = ['ph_value', 'turbidity_value', 'tds_value', 'temperature_value']
auditoria = pd.DataFrame({
    'nulos': leituras[sensor_cols].isna().sum(),
    'zeros': leituras[sensor_cols].eq(0).sum(),
    'minimo': leituras[sensor_cols].min(),
    'maximo': leituras[sensor_cols].max(),
    'distintos': leituras[sensor_cols].nunique(),
})
print('Dispositivos:', leituras['device_id'].value_counts().to_dict())
print('Horários duplicados:', leituras.duplicated(['device_id', 'data_hora']).sum())
print('Intervalo:', leituras['data_hora'].min(), 'até', leituras['data_hora'].max())
display(auditoria)

## 4. Divisão temporal e linha de base natural

O corte e a proporção são os mesmos documentados. O trecho final não participa do ajuste desta avaliação.

In [ ]:
avaliacao = temporal_validation(
    leituras, reference_end='2026-11-04T00:00:00Z',
    train_fraction=0.8, contamination=0.01,
)
resumo_temporal = pd.Series(avaliacao.summary, name='valor').to_frame()
display(resumo_temporal)
assert avaliacao.training['data_hora'].max() < avaliacao.validation['data_hora'].min()
print('Barreira temporal confirmada.')

## 5. Testes por intensidade

`detection_rate` é a taxa sinalizada após a perturbação artificial. `lift_vs_natural` mostra o aumento em relação aos dados não perturbados. Nenhuma das duas é acurácia sanitária.

In [ ]:
resultados = evaluate_perturbation_grid(avaliacao)
criterios = acceptance_criteria(resultados)
display(resultados.style.format({
    'detection_rate': '{:.1%}', 'lift_vs_natural': '{:+.1%}',
    'median_score': '{:.4f}', 'p90_score': '{:.4f}',
}))
display(criterios.style.format({'observed': '{:.1%}', 'target': '{:.1%}'}))
print('Critérios aprovados:', int(criterios['passed'].sum()), 'de', len(criterios))

In [ ]:
single = resultados.query("scenario == 'single_sensor'")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for sensor, grupo in single.groupby('sensor'):
    axes[0].plot(grupo['magnitude'], grupo['detection_rate'], marker='o', label=sensor)
axes[0].axhline(0.5, color='red', linestyle='--', label='meta forte: 50%')
axes[0].set(title='Resposta por sensor e intensidade', xlabel='Magnitude inserida', ylabel='Taxa de detecção')
axes[0].legend()
multi = resultados.query("scenario == 'multivariate'")
axes[1].plot(multi['magnitude'], multi['detection_rate'], marker='o', color='#6d5dfc')
axes[1].axhline(0.7, color='orange', linestyle='--', label='meta moderada: 70%')
axes[1].axhline(0.9, color='red', linestyle='--', label='meta forte: 90%')
axes[1].set(title='Resposta multivariada', xlabel='Proporção da perturbação forte', ylabel='Taxa de detecção')
axes[1].legend()
for ax in axes: ax.set_ylim(0, 1.05); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

## 6. Decisão provisória e exportação

Falhar em qualquer critério mantém o modelo como prova de conceito e indica exatamente onde recalibrar. Não altere as metas após visualizar o resultado.

In [ ]:
from pathlib import Path
import json

OUT = Path('/content/resultados_validacao_ia_sensores')
OUT.mkdir(exist_ok=True)
resultados.to_csv(OUT / 'grade_perturbacoes.csv', index=False)
criterios.to_csv(OUT / 'criterios_aceitacao.csv', index=False)
resumo_serializavel = {k: (v.isoformat() if hasattr(v, 'isoformat') else v) for k, v in avaliacao.summary.items()}
(OUT / 'resumo_temporal.json').write_text(json.dumps(resumo_serializavel, indent=2), encoding='utf-8')
status = 'APROVADO NOS CRITERIOS OFFLINE' if criterios['passed'].all() else 'MANTER COMO EXPLORATORIO E REVISAR FALHAS'
print(status)
!cd /content && zip -qr resultados_validacao_ia_sensores.zip resultados_validacao_ia_sensores
print('Arquivo gerado: /content/resultados_validacao_ia_sensores.zip')

## 7. Ensaio físico — executar depois

Use `docs/metodologia/eventos_controlados_sensores.example.json` para registrar as intervenções. Após o experimento, carregue o arquivo e aplique `evaluate_controlled_events` às leituras pontuadas pelo artefato final. Não execute esta etapa sem eventos reais registrados.